# 27 - Robot Safety And Reliability

## What / Why / How

**What we are trying to do:** Implement basic safety layers: action limits, a watchdog, barrier-like filtering, and hazard logging.

**Why this matters:** Robots can damage hardware and hurt people. Safety must be independent of the learned policy or main controller.

**How we will do it:** Filter unsafe proposed velocities, simulate command timeouts, log safety interventions, and visualize constrained motion.

## Goal

Safety is not a final checkbox. It is part of the architecture.

You will implement:

- Action limits
- A watchdog
- A simple control barrier function
- Hazard logging

In [ ]:
import math
import random
import sys
from pathlib import Path

import numpy as np

ROOT = Path.cwd()
if not (ROOT / "src").exists() and (ROOT.parent / "src").exists():
    ROOT = ROOT.parent
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

try:
    import matplotlib.pyplot as plt
    HAS_PLOT = True
except ModuleNotFoundError:
    plt = None
    HAS_PLOT = False

np.set_printoptions(precision=3, suppress=True)

def plot_unavailable():
    if not HAS_PLOT:
        print("Install matplotlib to see plots: pip install -r requirements.txt")

In [ ]:
from robotics_mastery.safety import limit_norm

def safety_filter(position, proposed_velocity, obstacle, min_distance=0.5):
    p = np.asarray(position)
    v = limit_norm(proposed_velocity, 0.4)
    vector_from_obstacle = p - obstacle
    distance = np.linalg.norm(vector_from_obstacle)
    if distance < min_distance:
        outward = vector_from_obstacle / (distance + 1e-9)
        if v @ outward < 0:
            v = v - (v @ outward) * outward
    return v

obstacle = np.array([0.0, 0.0])
position = np.array([0.3, 0.0])
proposed = np.array([-0.3, 0.0])
print("proposed:", proposed, "filtered:", safety_filter(position, proposed, obstacle))

In [ ]:
class Watchdog:
    def __init__(self, timeout_steps):
        self.timeout_steps = timeout_steps
        self.last_command_step = -1

    def command_received(self, step):
        self.last_command_step = step

    def ok(self, step):
        return step - self.last_command_step <= self.timeout_steps

watchdog = Watchdog(timeout_steps=3)
for step in range(8):
    if step in [0, 1, 2]:
        watchdog.command_received(step)
    print(step, "motors enabled?", watchdog.ok(step))

In [ ]:
hazards = []
pos = np.array([-1.0, 0.0])
goal = np.array([1.0, 0.0])
route = [pos.copy()]
for step in range(80):
    proposed = 0.6 * (goal - pos)
    filtered = safety_filter(pos, proposed, obstacle)
    if np.linalg.norm(filtered - proposed) > 1e-6:
        hazards.append((step, "action_filtered", pos.copy()))
    pos = pos + 0.05 * filtered
    route.append(pos.copy())
route = np.array(route)
print("hazard events:", len(hazards))
print("final pos:", route[-1])

In [ ]:
if HAS_PLOT:
    plt.figure(figsize=(5, 4))
    plt.plot(route[:, 0], route[:, 1])
    plt.scatter([obstacle[0]], [obstacle[1]], c="tab:red", label="obstacle")
    plt.gca().add_patch(plt.Circle(obstacle, 0.5, color="tab:red", alpha=0.15))
    plt.axis("equal")
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.title("Safety-filtered motion")
    plt.show()
else:
    plot_unavailable()

## Exercises

1. Add speed zones near people.
2. Add current/torque limits.
3. Write an emergency-stop test plan.
4. Explain why a learned policy should not be the only safety layer.